# Prediction

Use existing databases to make predictions and perform interpolation.

In [1]:
import pprint

import cutcutcodec

from mendevi.models.nn import DNN
from mendevi.models.gpr import GPR
from mendevi.models.lr import LR

In [2]:
dataset = "svtav1_vs_rav1e_vs_aom.db"  # the dataset name, automatically downloaded
val = 0.2  # keep 80% of data for training, 20% for validation
video = cutcutcodec.utils.get_project_root() / "media" / "video" / "intro.webm"

## Biaised linear regression

Predicts the output as a linear combination of the inputs, plus a constant.
It is a very fast model $O(n)$.

In [3]:
# model training
lr_model = LR(
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=["encoder"],  # optional, this feature provides one model per encoder
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster encoder=libaom-av1...
█ █ this cluster contains 2428 observations
█ █ 5.24 s
█ Fit on cluster encoder=librav1e...
█ █ this cluster contains 2473 observations
█ █ 66.06 ms
█ Fit on cluster encoder=libsvtav1...
█ █ this cluster contains 2472 observations
█ █ 67.85 ms
█ 2 min 15 s


In [4]:
# validation from dataset
lr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")
# print(lr_model.accuracy)  # for each cluster name, associate for each output label, the predicted and validation data

Validate 'Linear Regression for a predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster encoder=libaom-av1...
█ █ this cluster contains 596 observations
█ █ 35.79 ms
█ Validate on cluster encoder=librav1e...
█ █ this cluster contains 615 observations
█ █ 55.92 ms
█ Validate on cluster encoder=libsvtav1...
█ █ this cluster contains 613 observations
█ █ 33.01 ms
█ 1 min 12 s
MAPE of log_rate: 3.54%
RMS  of log_rate: 0.2951
MAPE of psnr: 11.65%
RMS  of psnr: 6.709
MAPE of ssim: 4.17%
RMS  of ssim: 0.05084
MAPE of rate: 66.81%
RMS  of rate: 1.593e+07


In [5]:
# prediction from a video
pred = lr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Linear Regression for a predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 11.71 ms
█ 214.21 ms


{'log_rate': [6.372843265533447],
 'psnr': [34.86414337158203],
 'ssim': [0.9203274846076965]}


## Gaussian process regression

Very slow model $O(n^3)$.

In [6]:
# model training
gpr_model = GPR(  # no aggregation because it fits only on continuous parameters
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= .5",  # two slow on all data
)

Fit 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster effort=slow, encoder=libaom-av1...
█ █ this cluster contains 509 observations
/home/robin/.pyenv/versions/3.15/lib/python3.15/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
█ █ 1 min 58 s
█ Fit on cluster effort=slow, encoder=librav1e...
█ █ this cluster contains 555 observations
█ █ 2 min 37 s
█ Fit on cluster effort=medium, encoder=librav1e...
█ █ this cluster contains 518 observations
/home/robin/.pyenv/versions/3.15/lib/python3.15/site-packages/sklearn/gaussian_process/kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling f

In [7]:
# validation from dataset
gpr_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'Gaussian Process Regression predictive model'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster effort=fast, encoder=libaom-av1...
█ █ this cluster contains 201 observations
█ █ 22.85 ms
█ Validate on cluster effort=fast, encoder=librav1e...
█ █ this cluster contains 181 observations
█ █ 29.74 ms
█ Validate on cluster effort=slow, encoder=librav1e...
█ █ this cluster contains 208 observations
█ █ 21.48 ms
█ Validate on cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 196 observations
█ █ 22.61 ms
█ Validate on cluster effort=medium, encoder=libaom-av1...
█ █ this cluster contains 202 observations
█ █ 19.67 ms
█ Validate on cluster effort=medium, encoder=librav1e...
█ █ this cluster contains 226 observations
█ █ 23.91 ms
█ Validate on cluster effort=fast, encoder=libsvtav1...
█ █ this cluster contains 212 observations
█ █ 33.17 ms
█ Validate on cluster effort=slow, encoder=libsvtav1...
█ █ this cluster contains 205 observations
█ █ 48.92 ms


In [8]:
# prediction from a video
pred = gpr_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'Gaussian Process Regression predictive model'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster effort=medium, encoder=libsvtav1...
█ █ this cluster contains 1 observations
█ █ 9.11 ms
█ 45.17 ms


{'log_rate': [6.325233459472656],
 'psnr': [39.73919677734375],
 'ssim': [0.9459365010261536]}


## Deep neuronal network

Medium speed model, $O(n)$.

In [9]:
# model training
dnn_model = DNN(  # see parameters niter, learning_rate and total
    input_labels=["effort", "encoder", "pixels", "quality", "rms_sobel", "rms_time_diff"],
    aggregation=[],  # optional, this feature force a single common model
    output_labels=["log_rate", "psnr", "ssim"],
).fit(
    dataset,
    table="t_enc_encode",  # optional, help to find the good SQL request
    select=f"uniform(vid_hash) >= {val}",
)

Fit 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Fit on cluster ...
█ █ this cluster contains 7373 observations
█ █ 8 min 0 s                                                                                                        
█ 8 min 35 s


In [10]:
# validation from dataset
dnn_model.validate(dataset, table="t_enc_encode", select=f"uniform(vid_hash) < {val}")

Validate 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ get SQL extractor
█ perform SQL query
█ Validate on cluster ...
█ █ this cluster contains 1824 observations
█ █ 74.01 ms
█ 39.83 s
MAPE of log_rate: 2.81%
RMS  of log_rate: 0.2254
MAPE of psnr: 6.84%
RMS  of psnr: 3.824
MAPE of ssim: 2.30%
RMS  of ssim: 0.03144
MAPE of rate: 44.23%
RMS  of rate: 1.367e+07


In [11]:
# prediction from a video
pred = dnn_model.predict_from_video(video, effort="medium", encoder="libsvtav1", quality=0.5)
pprint.pprint(pred)

Predict 'regressive model to predict log_rate, psnr, ssim from effort, encoder, pixels, quality, rms_sobel, rms_time_diff'...
█ check args
█ vectorize args
█ predict
█ Predict for cluster ...
█ █ this cluster contains 1 observations
█ █ 4.60 ms
█ 37.76 ms


{'log_rate': [], 'psnr': [], 'ssim': []}
